    # 📘 Chapter 1: Joins & Set Operations (Airline Domain)

---

## 🎯 Chapter Objective

In this chapter, you will learn how to:

- Combine data across multiple airline-related tables
- Understand **when and why** different JOIN types are used
- Apply **set operations** to compare datasets
- Think about joins from a **business and production perspective**

By the end of this chapter, you should be comfortable answering:
> “How do I correctly combine airline operational data to answer real business questions?”

---

## 🧠 Business Context (Why Joins Matter in Airlines)

In the airline domain, data is **never stored in a single table**.

Typical entities include:
- Flights
- Airports
- Aircraft
- Carriers
- Passengers
- Schedules

To answer even a simple business question like:
> *“Which airline had the highest average delay last month?”*

You **must join multiple tables correctly**.

Incorrect joins lead to:
- Wrong metrics
- Double counting
- Business mistrust in data

---

## 1️⃣ Understanding the Base Tables

Before writing joins, always understand:

- **Primary keys**
- **Foreign keys**
- **Data grain**

📏 **Data Grain Reminder**  
At this stage, we usually deal with:
- One row per **flight**
- One row per **airport**
- One row per **carrier**

Knowing this prevents incorrect joins.

---

## 2️⃣ INNER JOIN — The Most Common Join

### 🧠 Business Use Case
> The operations team wants flight details along with airport information.

### Example
```sql
SELECT
    f.flight_id,
    f.flight_date,
    a.airport_name,
    a.city
FROM flights f
INNER JOIN airports a
    ON f.origin_airport = a.airport_code;
```
### 🧠 How to Think About This Join

- Start with the fact table (flights)

- Join to a dimension table (airports)

- Use a stable, unique key

### 📏 Data Grain
One row per flight.

### ⚠️ Production Note
INNER JOIN removes records with missing airport mappings.

## 3️⃣ LEFT JOIN — Preserving Business Truth
### 🧠 Business Use Case

Show all flights, even if airport details are missing.
```SQL
SELECT
    f.flight_id,
    f.flight_date,
    a.airport_name
FROM flights f
LEFT JOIN airports a
    ON f.origin_airport = a.airport_code;
```
### 🧠 Why LEFT JOIN Matters

- Ensures no flight is lost

- Highlights data quality gaps

### 📏 Data Grain
One row per flight.

### ⚠️ Production Note
NULL values after a LEFT JOIN often indicate:

- Late-arriving dimension data

- Reference data issues

## 4️⃣ RIGHT JOIN — Rare but Possible

In practice, RIGHT JOIN is rarely used.

Most engineers prefer:

- Rewriting logic using LEFT JOIN

- Keeping the fact table on the left

### 🎤 Interview Tip
If asked about RIGHT JOIN, explain why LEFT JOIN is preferred.

## 5️⃣ FULL OUTER JOIN — Data Reconciliation
### 🧠 Business Use Case

Compare scheduled flights vs actual flights to find mismatches.
```sql
SELECT
    s.flight_id AS scheduled_flight,
    a.flight_id AS actual_flight
FROM scheduled_flights s
FULL OUTER JOIN actual_flights a
    ON s.flight_id = a.flight_id;
```
### 🧠 How to Think About It

- FULL JOIN is used for comparison

- Common in reconciliation and audits

### ⚠️ Production Note
FULL JOINs are expensive on large datasets.

## 6️⃣ Set Operations — Comparing Datasets

Set operations treat query results as sets.

### UNION

Combines results and removes duplicates.

```SQL
SELECT flight_id FROM domestic_flights
UNION
SELECT flight_id FROM international_flights;
```
### UNION ALL

Keeps duplicates (faster and more common).
```SQL
SELECT flight_id FROM domestic_flights
UNION ALL
SELECT flight_id FROM international_flights;

```
### INTERSECT

Finds common records.
```sql
SELECT flight_id FROM delayed_flights
INTERSECT
SELECT flight_id FROM cancelled_flights;
```
### EXCEPT

Finds differences.
```sql
SELECT flight_id FROM scheduled_flights
EXCEPT
SELECT flight_id FROM actual_flights;
```

### 7️⃣ Joins vs Set Operations — Key Difference
| Aspect   | Joins           | Set Operations |
| -------- | --------------- | -------------- |
| Purpose  | Combine columns | Compare rows   |
| Output   | Wider rows      | Same columns   |
| Keys     | Required        | Not required   |
| Use case | Enrichment      | Reconciliation |

## 🔑 Key Insights (Chapter Summary)

- Always identify data grain before joining

- INNER JOIN filters data — LEFT JOIN preserves it

- LEFT JOIN is the most common join in analytics

- FULL JOIN is mainly for reconciliation

- Set operations compare datasets, not columns

- UNION ALL is preferred over UNION for performance

- Incorrect joins are a top cause of wrong KPIs


# 📘 Chapter 2: Aggregations & Grouping (Airline Domain)

---

## 🎯 Chapter Objective

In this chapter, you will learn how to:

- Use aggregation functions to compute airline KPIs
- Group data correctly based on business questions
- Avoid common aggregation mistakes that lead to wrong metrics
- Understand how aggregations behave in real production data

By the end of this chapter, you should confidently answer:
> “How do airlines measure performance using SQL?”

---

## 🧠 Business Context (Why Aggregations Matter)

Airlines operate on **metrics**, not raw data.

Business teams track:
- Average delay
- Cancellation rate
- Flights per route
- Aircraft utilization
- Daily / monthly performance trends

All of these are derived using **aggregations and grouping**.

Incorrect aggregation logic leads to:
- Wrong KPIs
- Bad operational decisions
- Loss of trust in analytics

---

## 1️⃣ Basic Aggregation Functions

SQL provides core aggregation functions:

- `COUNT`
- `SUM`
- `AVG`
- `MIN`
- `MAX`

### 🧠 Business Use Case
> Count the total number of flights operated.

```sql
SELECT
    COUNT(*) AS total_flights
FROM flights;
```
### 📏 Data Grain
Single row representing total flights.

### ⚠️ Production Note
COUNT(*) includes rows with NULLs in columns.

## 2️⃣ Aggregation with GROUP BY
### 🧠 Business Use Case

Number of flights per airline.
```sql
SELECT
    carrier_code,
    COUNT(*) AS flight_count
FROM flights
GROUP BY carrier_code;
```
### 📏 Data Grain
One row per carrier.

### 🧠 How to Think About GROUP BY

Columns in SELECT must either be:

- Aggregated, or

- Present in GROUP BY

GROUP BY defines the output grain
## 3️⃣ Aggregating with Business Meaning
### 🧠 Business Use Case

Average arrival delay per airline.
```sql
SELECT
    carrier_code,
    AVG(arrival_delay) AS avg_arrival_delay
FROM flights
WHERE flight_status = 'ARRIVED'
GROUP BY carrier_code;
```
### 📏 Data Grain
One row per carrier.

### ⚠️ Production Note
Exclude cancelled flights to avoid skewing averages.

## 4️⃣ Multiple Columns in GROUP BY
### 🧠 Business Use Case

Daily flight count per airline.
```sql
SELECT
    flight_date,
    carrier_code,
    COUNT(*) AS daily_flights
FROM flights
GROUP BY flight_date, carrier_code;
```
### 📏 Data Grain
One row per carrier per day.

### 🧠 Key Concept

Adding more columns to GROUP BY:

- cIncreases result granularity

- Increases row count

## 5️⃣ HAVING — Filtering After Aggregation
### 🧠 Business Use Case

Identify airlines with more than 1,000 flights.
```sql
SELECT
    carrier_code,
    COUNT(*) AS flight_count
FROM flights
GROUP BY carrier_code
HAVING COUNT(*) > 1000;
```
### 🧠 WHERE vs HAVING
| Clause | When it applies    |
| ------ | ------------------ |
| WHERE  | Before aggregation |
| HAVING | After aggregation  |
### 🎤 Interview Tip
This difference is asked very frequently.
## 6️⃣ Aggregations with NULL Handling
### 🧠 Business Use Case

Calculate average delay while handling NULLs.
```sql
SELECT
    carrier_code,
    AVG(COALESCE(arrival_delay, 0)) AS avg_delay
FROM flights
GROUP BY carrier_code;
```
### ⚠️ Production Warning
Replacing NULLs with 0 can distort metrics.

### Better approach:

- Understand why NULL exists

- Filter or handle based on business rules

## 7️⃣ Aggregation Pitfalls (Common Mistakes)
### ❌ Mistake 1: Aggregating at the Wrong Grain

- Aggregating per flight instead of per route

### ❌ Mistake 2: Forgetting Business Filters

- Including cancelled flights in delay metrics

### ❌ Mistake 3: Over-grouping

- Grouping by unnecessary columns

### Key Insight:
Wrong aggregation is worse than no aggregation.

## 🔑 Key Insights (Chapter Summary)

- Aggregations convert raw data into business metrics

- GROUP BY defines the output grain — always be explicit

- WHERE filters rows before aggregation

- HAVING filters groups after aggregation

- Always align aggregation logic with business rules

- NULL handling can drastically change results

- Most airline KPIs are aggregation-driven

# 📘 Chapter 3: Window Functions (Airline Domain)

---

## 🎯 Chapter Objective

In this chapter, you will learn how to:

- Understand what window functions are and why they are needed
- Differentiate between aggregation and window functions
- Use ranking, analytical, and offset window functions
- Solve real airline business problems using window functions
- Avoid common mistakes in window function usage

By the end of this chapter, you should confidently answer:
> “When should I use window functions instead of GROUP BY?”

---

## 🧠 Business Context (Why Window Functions Matter)

In airline analytics, we often need to answer questions like:

- What is the **top delayed flight per route**?
- How does today’s delay compare to **yesterday’s delay**?
- What is the **running total of flights per day**?
- Rank airlines by performance **without collapsing data**

These problems **cannot be solved correctly with GROUP BY alone**.

Window functions allow us to:
- Perform calculations **across related rows**
- Retain **row-level detail**
- Compute rankings, trends, and comparisons

---

## 1️⃣ Aggregations vs Window Functions

### ❌ Aggregation (GROUP BY)
- Collapses rows
- Loses row-level detail

```sql
SELECT
    carrier_code,
    AVG(arrival_delay) AS avg_delay
FROM flights
GROUP BY carrier_code;
```
📏 Data Grain
One row per carrier.

### ✅ Window Function

- Preserves rows

- Adds analytical insight

```sql
SELECT
    flight_id,
    carrier_code,
    AVG(arrival_delay) OVER (PARTITION BY carrier_code) AS avg_delay
FROM flights;
```
### 📏 Data Grain
One row per flight.

### Key Insight:

Window functions add information, they do not reduce rows.

## 2️⃣ Basic Window Function Syntax
```sql
<function>() OVER (
    PARTITION BY <columns>
    ORDER BY <columns>
)
```
### Components:

- PARTITION BY → Defines the window (grouping)

- ORDER BY → Defines the order inside the window

⚠️ ORDER BY is optional, but required for ranking and offset functions.

## 3️⃣ Ranking Functions (Most Common Use Case)
Available Ranking Functions:

- ROW_NUMBER()

- RANK()

- DENSE_RANK()

## 4️⃣ ROW_NUMBER() — Unique Ranking
### 🧠 Business Use Case

Find the most delayed flight per route.
```sql
SELECT *
FROM (
    SELECT
        flight_id,
        route,
        arrival_delay,
        ROW_NUMBER() OVER (
            PARTITION BY route
            ORDER BY arrival_delay DESC
        ) AS rn
    FROM flights
) t
WHERE rn = 1;
```
📏 Data Grain
One row per route.

### Key Insight:

ROW_NUMBER() assigns unique ranks, even if values are equal.

## 5️⃣ RANK() vs DENSE_RANK()
### 🧠 Business Use Case

Rank airlines by average delay.
```sql
SELECT
    carrier_code,
    AVG(arrival_delay) AS avg_delay,
    RANK() OVER (
        ORDER BY AVG(arrival_delay) DESC
    ) AS delay_rank
FROM flights
GROUP BY carrier_code;
```
### Difference Explained:
| Function   | Behavior            |
| ---------- | ------------------- |
| RANK       | Skips ranks on ties |
| DENSE_RANK | No gaps in ranking  |

### Key Insight:

Use DENSE_RANK() when you want continuous rankings.

## 6️⃣ ORDER BY Inside Window Functions
### 🧠 Business Use Case

Compare flight delays over time.
```sql
SELECT
    flight_id,
    flight_date,
    arrival_delay,
    AVG(arrival_delay) OVER (
        ORDER BY flight_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_avg_delay
FROM flights;
```
📏 Data Grain
One row per flight per date.

### Key Insight:

This enables running metrics, impossible with GROUP BY.

## 7️⃣ Offset Functions (LAG & LEAD)
### 🧠 Business Use Case

Compare today’s delay with previous day.
```sql
SELECT
    flight_date,
    arrival_delay,
    LAG(arrival_delay) OVER (
        ORDER BY flight_date
    ) AS previous_day_delay
FROM flights;
```

### LEAD Example
```sql
SELECT
    flight_date,
    arrival_delay,
    LEAD(arrival_delay) OVER (
        ORDER BY flight_date
    ) AS next_day_delay
FROM flights;
```
### Key Insight:

LAG and LEAD are essential for trend and variance analysis.

## 8️⃣ PARTITION BY with LAG / LEAD
### 🧠 Business Use Case

Compare delays per airline over time.
```sql
SELECT
    carrier_code,
    flight_date,
    arrival_delay,
    LAG(arrival_delay) OVER (
        PARTITION BY carrier_code
        ORDER BY flight_date
    ) AS prev_delay
FROM flights;
```
📏 Data Grain
One row per carrier per day.

## 9️⃣ Common Window Function Mistakes
### ❌ Mistake 1: Missing PARTITION BY

- Results calculated across entire dataset unintentionally

### ❌ Mistake 2: Incorrect ORDER BY

- Produces meaningless rankings

### ❌ Mistake 3: Using GROUP BY when window is needed

- Loss of row-level detail

### Key Insight:

If you need both detail and aggregation, you need a window function.

## 🔑 Key Insights (Chapter Summary)

- Window functions do not reduce rows

- Use GROUP BY for summaries, window functions for analytics

- PARTITION BY defines logical groups

- ORDER BY defines analytical sequence

- ROW_NUMBER, RANK, DENSE_RANK solve ranking problems

- LAG and LEAD enable trend analysis

- This chapter is critical for interviews and real-world analytics